## Install Libraries

In [ ]:
!pip install transformers peft torch gymnasium

In [ ]:
!pip install "trl<=0.11.0" --upgrade

In [ ]:
!pip install "git+https://github.com/volatilemolotov/agent-sandbox.git@gymnasium-api#subdirectory=clients/python/agentic-sandbox-client" --upgrade

## Example usage

In [5]:
from k8s_agent_sandbox.sandbox_env import SandboxEnv, SparseTaskReward, StepPenaltyReward

env = SandboxEnv(
    reward_fn=StepPenaltyReward(
        base=SparseTaskReward(
            success_fn=lambda obs, info: "manage.py" in obs and info["exit_code"] == 0
        ),
        penalty=0.02,
    ),
    warmpool="simple-sandbox-warmpool",
    connection_mode="gateway",   # swap to "gateway" for GKE
    connection_cfg={
        "gateway_name": "external-http-gateway",
        "gateway_namespace": "default",
    },
    max_episode_steps=15,
)

agent_tasks = [
    "create a django poll project",
]

trajectory = []

for task in agent_tasks:
    obs, info = env.reset(options={"task": task})
    terminated, truncated = False, False

    while not terminated and not truncated:
        action = "ls -la" #agent.act(obs, task)
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"""
obs={obs}
reward={reward}
terminated={terminated}
truncated={truncated}
info={info}
""")
        trajectory.append((obs, action, reward, info))

env.close()



obs=total 13
drwxr-xr-x 1 pwuser pwuser 4096 Jun 30 17:14 .
drwxr-xr-x 1 root   root   4096 Jun 30 17:14 ..
drwxr-xr-x 2 pwuser pwuser 4096 Jun 30 17:14 __pycache__
-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py

reward=-0.02
terminated=False
truncated=False
info={'exit_code': 0, 'stdout': 'total 13\ndrwxr-xr-x 1 pwuser pwuser 4096 Jun 30 17:14 .\ndrwxr-xr-x 1 root   root   4096 Jun 30 17:14 ..\ndrwxr-xr-x 2 pwuser pwuser 4096 Jun 30 17:14 __pycache__\n-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py\n', 'stderr': '', 'elapsed_ms': 126, 'step': 1}


obs=total 13
drwxr-xr-x 1 pwuser pwuser 4096 Jun 30 17:14 .
drwxr-xr-x 1 root   root   4096 Jun 30 17:14 ..
drwxr-xr-x 2 pwuser pwuser 4096 Jun 30 17:14 __pycache__
-rw-r--r-- 1 pwuser pwuser 1023 Jun 26 09:45 main.py

reward=-0.02
terminated=False
truncated=False
info={'exit_code': 0, 'stdout': 'total 13\ndrwxr-xr-x 1 pwuser pwuser 4096 Jun 30 17:14 .\ndrwxr-xr-x 1 root   root   4096 Jun 30 17:14 ..\ndrwxr-xr-x 2 pwuser pwuser 4

## Fine-tune an LLM

In [52]:
import torch
import os
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from peft import LoraConfig

from k8s_agent_sandbox.sandbox_env import SandboxEnv, SparseTaskReward

# ── 1. Configuration & Setup ─────────────────────────────────────────────────

model_name = "Qwen/Qwen2.5-Coder-1.5B"  # Good starting point for code/bash
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=4,           # Trigger an update every 4 steps
    mini_batch_size=1,      # Must be a divisor of batch_size
    # gradient_accumulation_steps=4 # You can use this if VRAM is tight
)

# Apply LoRA to make training memory-efficient
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

# ── 2. Load Model and Tokenizer ──────────────────────────────────────────────

# TRL uses a special wrapper that adds a "Value Head" to the LLM for PPO
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    peft_config=lora_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Initialize the PPO Trainer
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=None, # TRL handles the reference model internally with PEFT
    tokenizer=tokenizer,
)

# ── 3. Initialize Your Custom Environment ────────────────────────────────────

# Let's use a task where the agent must create a specific file
env = SandboxEnv(
    reward_fn=SparseTaskReward(
        success_fn=lambda obs, info: "hello_world.sh" in obs and info["exit_code"] == 0
    ),
    connection_mode="gateway",
    connection_cfg={
        "gateway_name": "external-http-gateway",
        "gateway_namespace": "default",
    },
    warmpool="simple-sandbox-warmpool",
    max_episode_steps=5,
)

# ── 4. The Training Loop ─────────────────────────────────────────────────────

epochs = 50
task = "Create a bash script named hello_world.sh that prints 'Hello World'."

# 1. Global buffers to hold data across episodes
queries_buffer = []
responses_buffer = []
rewards_buffer = []

for epoch in range(epochs):
    obs, info = env.reset(options={"task": task})
    terminated, truncated = False, False
    
    print(f"\n--- Starting Epoch {epoch} ---")

    while not terminated and not truncated:
        prompt = f"""System: You are an expert sh programmer. 
You run commands only in this format: sh -c \"<YOUR TARGET COMMAND> && ls\".
Task: {task}
Terminal Output: {obs}
Command to run:"""
        query_tensor = tokenizer(prompt, return_tensors="pt").input_ids.squeeze().to(ppo_trainer.accelerator.device)
        
        generation_kwargs = {"max_new_tokens": 32, "do_sample": True, "top_k": 0.0, "top_p": 1.0}
        response_tensor = ppo_trainer.generate(query_tensor, **generation_kwargs).squeeze()
        
        # Only take the newly generated tokens
        action_tensor = response_tensor[len(query_tensor):]
        action = tokenizer.decode(action_tensor, skip_special_tokens=True).strip()
        print(f"Agent executed: {action}")
        
        next_obs, reward, terminated, truncated, info = env.step(action)
        print(f"Reward: {reward} | Exit Code: {info['exit_code']}")
        
        # 2. Append the tensors to our global buffer
        queries_buffer.append(query_tensor)
        responses_buffer.append(action_tensor)
        rewards_buffer.append(torch.tensor(reward, dtype=torch.float32))
        
        # 3. Check if the buffer is full
        if len(queries_buffer) == ppo_config.batch_size:
            print(">> Buffer full! Executing PPO optimization step...")
            stats = ppo_trainer.step(queries_buffer, responses_buffer, rewards_buffer)
            print(f">> PPO Loss: {stats['ppo/loss/total']}")
            
            # Clear the buffer for the next batch
            queries_buffer = []
            responses_buffer = []
            rewards_buffer = []
        
        obs = next_obs
        
        if reward > 0:
            print("Goal reached! Ending episode early.")
            break

env.close()

# Save your fine-tuned model adapters!
path = "./sandbox-agent-lora"
if not os.path.exists(path):
    os.makedirs(path)
model.save_pretrained(path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



--- Starting Epoch 0 ---


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: sh -c "bash hello_world.sh && ls"


```bash
#!/bin/bash

echo "Hello World"
exit 0

```

```language-b
Reward: 0.0 | Exit Code: 127


...

>> PPO Loss: 0.047954484820365906


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: sh -c "sh -c 'sh -c \"sh -c "sh -c \"sh -c \"sh -c "sh -c \"
Reward: 0.0 | Exit Code: -1


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Agent executed: sh -c "echo "Hello World" && ls"
Reward: 0.0 | Exit Code: 0
Agent executed: sh -c "sh -c 'echo Hello && ls;' && ls"
Reward: 0.0 | Exit Code: 0


## Load and Test the Fine-tuned LLM

In [53]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ── 1. Configuration ─────────────────────────────────────────────────────────

base_model_name = "Qwen/Qwen2.5-Coder-1.5B"
adapter_path = "./sandbox-agent-lora"

# ── 2. Load Model and Tokenizer ──────────────────────────────────────────────

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    torch_dtype=torch.float16,  # Uses less VRAM during inference
)

print("Applying LoRA adapter...")
# Merge the fine-tuned adapter weights with the base model
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()  # Set model to evaluation mode

# ── 3. Prepare the Environment State ─────────────────────────────────────────

task = "Create a bash script named hello_world.sh that prints 'Hello World'."
# Assuming this is step 1, the initial observation from env.reset() is:
obs = "Sandbox ready." 

# CRITICAL: The prompt format must perfectly match how it was trained
prompt = f"""System: You are an expert sh programmer. 
You run commands only in this format: sh -c \"<YOUR TARGET COMMAND> && ls\".
Task: {task}
Terminal Output: {obs}
Command to run:"""

# ── 4. Generate the Action ───────────────────────────────────────────────────

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("Generating command...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,  # Greedy decoding ensures the most confident, deterministic output
        pad_token_id=tokenizer.eos_token_id
    )

# ── 5. Extract and Display the Command ───────────────────────────────────────

# Slice the output tensor to ignore the prompt and only decode the newly generated tokens
input_length = inputs.input_ids.shape[1]
generated_tokens = outputs[0][input_length:]

action = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n" + "="*40)
print(f"Prompt Context:\n{prompt}")
print("="*40)
print(f"Agent's Command: {action}")
print("="*40)

Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying LoRA adapter...
Generating command...

Prompt Context:
System: You are an expert sh programmer. 
You run commands only in this format: sh -c "<YOUR TARGET COMMAND> && ls".
Task: Create a bash script named hello_world.sh that prints 'Hello World'.
Terminal Output: Sandbox ready.
Command to run:
Agent's Command: sh -c "sh -c 'sh -c \"sh -c \"sh -c \"sh -c \"sh -c \"sh -c \"
